# Notebook 05: Decision Trees and Ensembles

## Why This Matters

Decision trees are the building blocks of the most competitive classical ML algorithms in production today: Random Forest, XGBoost, LightGBM, and CatBoost. Understanding how a single tree learns — split criteria, depth, pruning — is prerequisite to understanding why ensembles of hundreds of trees outperform them. Random Forest and Gradient Boosting are among the most commonly tested algorithms in applied ML interviews precisely because they win most tabular data competitions and appear in virtually every fraud, risk, and recommendation system. The key interview questions here are: why does bagging reduce variance, why does boosting reduce bias, and when do you choose one over the other.

## 1. Decision Tree Internals: Splits and Impurity

A decision tree partitions the feature space into axis-aligned rectangles by greedily selecting the split (feature, threshold) that maximally reduces **impurity** at each node.

For classification, two common impurity measures:

**Gini impurity**: $G = 1 - \sum_{k} p_k^2$ — probability of misclassification if a random label is drawn from the node distribution.

**Entropy**: $H = -\sum_k p_k \log_2 p_k$ — information content. Information gain = parent entropy − weighted child entropy.

For regression: **MSE** (variance reduction) — split that minimizes the weighted MSE of the two child nodes.

Both criteria produce similar trees in practice. Gini is slightly faster (no log), entropy can occasionally find better splits in imbalanced classes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

# Visualize impurity measures
p = np.linspace(0.001, 0.999, 300)
gini = 2 * p * (1 - p)  # binary Gini = 2*p*(1-p)
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
misclass = 1 - np.maximum(p, 1 - p)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, gini,    label='Gini', linewidth=2)
ax.plot(p, entropy / 2, label='Entropy / 2 (scaled)', linewidth=2)
ax.plot(p, misclass, label='Misclassification error', linewidth=2, linestyle='--')
ax.set_xlabel('p (fraction of positive class)')
ax.set_ylabel('Impurity')
ax.set_title('Impurity Measures for Binary Classification')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Implement information gain from scratch
def entropy(y):
    if len(y) == 0: return 0
    p = np.mean(y)
    if p in (0, 1): return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

def information_gain(y, y_left, y_right):
    n, nl, nr = len(y), len(y_left), len(y_right)
    return entropy(y) - (nl/n) * entropy(y_left) - (nr/n) * entropy(y_right)

# Find best split on a simple dataset
np.random.seed(42)
X_demo = np.random.randn(200, 2)
y_demo = (X_demo[:, 0] + X_demo[:, 1] > 0).astype(int)

# Try all thresholds on feature 0
thresholds = np.percentile(X_demo[:, 0], range(5, 100, 5))
gains = []
for t in thresholds:
    left_mask = X_demo[:, 0] <= t
    gains.append(information_gain(y_demo, y_demo[left_mask], y_demo[~left_mask]))

best_t = thresholds[np.argmax(gains)]
print(f"Best threshold on feature 0: {best_t:.3f}  (IG = {max(gains):.4f})")

plt.figure(figsize=(7, 3))
plt.plot(thresholds, gains, 'steelblue', marker='o', markersize=4)
plt.axvline(best_t, color='red', linestyle='--', label=f'Best threshold={best_t:.2f}')
plt.xlabel('Threshold on feature 0')
plt.ylabel('Information Gain')
plt.title('Information Gain vs. Split Threshold')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Tree Depth, Pruning, and Overfitting

A decision tree grown to full depth (until all leaves are pure) memorizes the training data perfectly — train accuracy = 100%, test accuracy far lower. This is the prototypical example of high variance.

Key hyperparameters to control overfitting:
- `max_depth`: maximum tree depth (most important)
- `min_samples_split`: minimum samples to split a node
- `min_samples_leaf`: minimum samples in any leaf
- `max_leaf_nodes`: maximum number of leaves
- `ccp_alpha`: cost-complexity pruning parameter (prunes branches that don't improve generalization)

In [ ]:
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

depths = range(1, 20)
train_accs, test_accs = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, t.predict(X_tr)))
    test_accs.append(accuracy_score(y_te, t.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_accs, 'steelblue', marker='o', markersize=4, label='Train accuracy')
plt.plot(depths, test_accs,  'coral',     marker='s', markersize=4, label='Test accuracy')
best_depth = depths[np.argmax(test_accs)]
plt.axvline(best_depth, color='gray', linestyle='--', label=f'Best depth={best_depth}')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Overfitting as Depth Increases')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Best test accuracy at depth={best_depth}: {max(test_accs):.4f}")

## 3. Bagging and Random Forest

**Bagging** (Bootstrap Aggregating) reduces variance by training multiple models on different bootstrap samples (random draw with replacement from training set) and averaging their predictions.

Why does averaging reduce variance? For i.i.d. estimators with variance σ², the average of n estimators has variance σ²/n. Trees are high-variance estimators — bagging exploits this directly.

**Random Forest** extends bagging with **feature randomization**: at each split, only a random subset of √p features are considered. This de-correlates the trees (bagged trees on the same features are still correlated), reducing the ensemble variance further.

The **out-of-bag (OOB) samples** — roughly 37% of data not drawn in each bootstrap — provide a free, unbiased estimate of test error without needing a separate validation set.

In [ ]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

# Single tree
single_tree = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)

# Bagging (no feature subsampling)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100, max_samples=1.0, max_features=1.0,
    bootstrap=True, oob_score=True, random_state=42
).fit(X_tr, y_tr)

# Random Forest (bagging + feature randomization)
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42).fit(X_tr, y_tr)

for name, model in [('Single Tree', single_tree), ('Bagging', bagging), ('Random Forest', rf)]:
    test_acc = accuracy_score(y_te, model.predict(X_te))
    oob = getattr(model, 'oob_score_', 'N/A')
    print(f"{name:15s} | Test acc={test_acc:.4f} | OOB score={oob}")

In [ ]:
# Convergence: test accuracy as number of trees grows
n_trees = [1, 5, 10, 25, 50, 100, 200, 300]
rf_accs = []
for n in n_trees:
    m = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_tr, y_tr)
    rf_accs.append(accuracy_score(y_te, m.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.semilogx(n_trees, rf_accs, 'steelblue', marker='o', markersize=5)
plt.xlabel('Number of trees')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Accuracy Converges as Trees Added')
plt.tight_layout()
plt.show()
print("Key insight: Random Forest converges and stops overfitting — more trees only help.")

## 4. Gradient Boosting: High-Level Intuition

**Boosting** builds an ensemble sequentially: each new tree is trained to correct the errors of the existing ensemble. The final prediction is a weighted sum of all trees.

**Gradient Boosting** frames this as gradient descent in function space. At each step, fit a new tree to the **negative gradient** of the loss with respect to the current predictions:

1. Initialize: $F_0(x) = \text{const}$
2. For m = 1 to M:
   - Compute pseudo-residuals: $r_i = -\partial L(y_i, F_{m-1}(x_i)) / \partial F_{m-1}(x_i)$
   - Fit a tree $h_m$ to $\{(x_i, r_i)\}$
   - Update: $F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)$ where ν is the learning rate

For squared-error loss, the pseudo-residuals are just the ordinary residuals $(y - \hat{y})$. For cross-entropy loss, they are $(y - \hat{p})$.

Key contrast with Random Forest:
- RF builds trees in **parallel** (bagging) → reduces variance
- GBM builds trees **sequentially** (boosting) → reduces bias
- GBM uses shallow trees (depth 3-8) that are individually weak; RF uses deep trees

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Implement gradient boosting from scratch (regression, squared error)
from sklearn.tree import DecisionTreeRegressor

class GradientBoostingFromScratch:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None
    
    def fit(self, X, y):
        self.F0 = np.mean(y)  # init with constant (mean)
        F = np.full(len(y), self.F0)
        for _ in range(self.n_estimators):
            residuals = y - F  # pseudo-residuals for MSE loss
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            F += self.lr * tree.predict(X)
            self.trees.append(tree)
        return self
    
    def predict(self, X):
        F = np.full(len(X), self.F0)
        for tree in self.trees:
            F += self.lr * tree.predict(X)
        return F

# Test on a regression problem
from sklearn.datasets import fetch_california_housing
house = fetch_california_housing()
Xh, yh = house.data[:2000], house.target[:2000]  # small subset for speed
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=42)

gbm_scratch = GradientBoostingFromScratch(n_estimators=100, learning_rate=0.1, max_depth=3)
gbm_scratch.fit(Xh_tr, yh_tr)
rmse_scratch = np.sqrt(np.mean((yh_te - gbm_scratch.predict(Xh_te))**2))

from sklearn.ensemble import GradientBoostingRegressor
gbm_sk = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbm_sk.fit(Xh_tr, yh_tr)
rmse_sk = np.sqrt(np.mean((yh_te - gbm_sk.predict(Xh_te))**2))

print(f"From-scratch GBM RMSE: {rmse_scratch:.4f}")
print(f"sklearn GBM RMSE:      {rmse_sk:.4f}")

## 5. Bias-Variance Decomposition: Forest vs. Boosting

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Compare RF and GBM across varying dataset sizes to see generalization curves
sizes = [50, 100, 200, 400, 800]
rf_scores, gbm_scores = [], []

X_bc_s = StandardScaler().fit_transform(X_bc)
for size in sizes:
    X_sub, _, y_sub, _ = train_test_split(X_bc_s, y_bc, train_size=size, random_state=42)
    for model, store in [
        (RandomForestClassifier(n_estimators=100, random_state=42), rf_scores),
        (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42), gbm_scores),
    ]:
        cv = cross_val_score(model, X_sub, y_sub, cv=5, scoring='roc_auc')
        store.append(cv.mean())

plt.figure(figsize=(8, 4))
plt.plot(sizes, rf_scores,  'steelblue', marker='o', label='Random Forest')
plt.plot(sizes, gbm_scores, 'coral',     marker='s', label='Gradient Boosting')
plt.xlabel('Training set size')
plt.ylabel('CV AUC')
plt.title('RF vs. GBM: Learning Curves')
plt.legend()
plt.tight_layout()
plt.show()

print("\nFull dataset comparison:")
for name, model in [
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('Single Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42)),
]:
    scores = cross_val_score(model, X_bc_s, y_bc, cv=5, scoring='roc_auc')
    print(f"{name:25s} | AUC = {scores.mean():.4f} ± {scores.std():.4f}")

## Summary

| Algorithm | Mechanism | Bias | Variance | Scale Sensitivity | Best Use Case |
|-----------|-----------|------|----------|-------------------|---------------|
| Decision Tree | Greedy splits | High (shallow) / Low (deep) | High (deep) | No | Interpretability; rule extraction |
| Bagging | Bootstrap + average | Same as base | Lower | No | Reducing variance of any estimator |
| Random Forest | Bagging + feature subsampling | Moderate | Low | No | Robust baseline; large feature sets |
| Gradient Boosting | Sequential residual fitting | Low | Moderate | No | Wins on tabular data; needs tuning |

| Question | Answer |
|----------|--------|
| Why does RF not overfit with more trees? | Averaging reduces variance; additional uncorrelated trees cannot increase bias |
| Why use shallow trees in GBM? | Each tree corrects small errors; deep trees would overfit the residuals |
| When would you choose RF over GBM? | Fewer hyperparameters; faster training; less risk of overfitting on small data |
| What is OOB error? | Error estimated on the ~37% of training samples not included in each bootstrap draw |